<a href="https://colab.research.google.com/github/asheldrick-research/ecsm-framework/blob/main/ECSM_Electron_Like_Packet_Radiative_Stack_Consistency_Audit_V30_CLEAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ECSM Electron-Like Packet Radiative Stack Consistency Audit — V30 CLEAN

**Run label:** `V30_radiative_stack_consistency_audit`

**Purpose:** This notebook does **not** introduce a new doorway. It audits whether the inherited electron-like packet radiative chain from **V25 through V29** remains mutually consistent under one shared set of invariants, no-fit flags, finite-response behaviour, and claim boundaries.

The audited chain is:

\[
\mathrm{V21b}\rightarrow\mathrm{V22}\rightarrow\mathrm{V23}\rightarrow\mathrm{V24}\rightarrow\mathrm{V25}\rightarrow\mathrm{V26}\rightarrow\mathrm{V27}\rightarrow\mathrm{V27b}\rightarrow\mathrm{V28}\rightarrow\mathrm{V29}\rightarrow\mathrm{V30}.
\]

V30 prepares the programme for the first strict **non-doorway observable test** by asking whether the existing doorway stack is coherent enough to freeze.


In [ ]:

# CELL 1 — SETUP / OUTPUT DIRECTORY

import os, json, math, zipfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/content") if Path("/content").exists() else Path("/mnt/data")
OUT_DIR = BASE_DIR / "ecsm_electron_like_radiative_stack_consistency_audit_v30_out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_LABEL = "V30_radiative_stack_consistency_audit"
FINAL_TARGET = "PASS_STRONG_RADIATIVE_STACK_CONSISTENCY_AUDIT"

print("Output directory:", OUT_DIR)



## 1. Inherited handoff

The V30 audit deliberately freezes the inherited packet values. The purpose is to check consistency, not to repair or refit the electron-like packet.


In [ ]:

# CELL 2 — INHERITED HANDOFF OBJECT

handoff = {
    "source_chain": "V21b -> V22 -> V23 -> V24 -> V25 -> V26 -> V27 -> V27b -> V28 -> V29",
    "q_eff": -0.9993035720499918,
    "E_rest_keV_from_V21b": 514.0698767738036,
    "electron_rest_energy_target_keV_comparison_only": 510.9989506917532,
    "alpha_fs_input": 0.0072973525692838015,
    "V22_claim": "coherent-limit free Dirac-like propagation recovered",
    "V23_claim": "minimal emergent gauge-coupling doorway recovered",
    "V24_claim": "Pauli-limit spin-field coupling and tree-level g=2 doorway recovered",
    "V25_claim": "Schwinger-scale one-loop magnetic-moment doorway recovered",
    "V26_claim": "finite radiative self-energy / propagator doorway recovered",
    "V27_claim": "finite Lamb-shift-like structural doorway recovered",
    "V27b_claim": "non-fitted leading-log Lamb-shift scale-closure doorway recovered",
    "V28_claim": "finite vacuum-polarisation / running-response doorway recovered",
    "V29_claim": "finite elastic scattering-amplitude doorway recovered",
    "V30_target": "radiative stack consistency audit",
    "electron_mass_derivation_claimed": False,
    "measured_observable_values_used_as_fits": False,
    "full_QED_claimed": False,
    "renormalised_S_matrix_claimed": False
}

with open(OUT_DIR / "v21b_to_v29_handoff_v30.json", "w") as f:
    json.dump(handoff, f, indent=2)

handoff



## 2. Core invariant audit

V30 rechecks the invariant packet values used across the sequence. The comparison electron rest energy remains a comparison-only number and is not used to repair the packet.


In [ ]:

# CELL 3 — CORE INVARIANT AUDIT

q_eff = handoff["q_eff"]
E_rest = handoff["E_rest_keV_from_V21b"]
E_electron_cmp = handoff["electron_rest_energy_target_keV_comparison_only"]
alpha_fs = handoff["alpha_fs_input"]

invariant_summary = {
    "q_eff": q_eff,
    "charge_error_abs_to_minus_one": abs(q_eff + 1.0),
    "E_rest_keV": E_rest,
    "electron_rest_energy_comparison_keV": E_electron_cmp,
    "fractional_E_rest_difference_to_comparison": abs(E_rest - E_electron_cmp) / E_electron_cmp,
    "alpha_fs_input": alpha_fs,
    "alpha_over_2pi_schwinger_scale": alpha_fs / (2.0 * math.pi),
    "invariants_finite": all(np.isfinite([q_eff, E_rest, E_electron_cmp, alpha_fs])),
    "electron_mass_repaired_or_refit": False
}

invariant_df = pd.DataFrame([
    {"quantity": k, "value": v}
    for k, v in invariant_summary.items()
])
invariant_df.to_csv(OUT_DIR / "core_invariant_summary_v30.csv", index=False)

with open(OUT_DIR / "core_invariant_summary_v30.json", "w") as f:
    json.dump(invariant_summary, f, indent=2)

print(json.dumps(invariant_summary, indent=2))
invariant_df


In [ ]:

# CELL 4 — INVARIANT VISUAL AUDIT

fig, ax = plt.subplots(figsize=(7.5, 4.5))
labels = ["|q_eff+1|", "|ΔE|/E_e", "α_fs", "α/2π"]
values = [
    invariant_summary["charge_error_abs_to_minus_one"],
    invariant_summary["fractional_E_rest_difference_to_comparison"],
    invariant_summary["alpha_fs_input"],
    invariant_summary["alpha_over_2pi_schwinger_scale"],
]
ax.bar(labels, values)
ax.set_yscale("log")
ax.set_ylabel("dimensionless value")
ax.set_title("V30 inherited invariant scale audit")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_v30_inherited_invariant_scale_audit.png", dpi=200)
plt.show()



## 3. Radiative doorway stack

The stack begins at V25 because V25 is the first one-loop radiative-scale doorway. V26–V29 then add self-energy/propagator, Lamb-like bound-state response, running response, and elastic scattering-amplitude behaviour.


In [ ]:

# CELL 5 — RADIATIVE DOORWAY STACK TABLE

radiative_stack = [
    {
        "version": "V25",
        "sector": "vertex / magnetic moment",
        "doorway_claim": "Schwinger-scale one-loop magnetic-moment doorway",
        "primary_scale_or_metric": alpha_fs / (2.0 * math.pi),
        "primary_metric_label": "alpha_fs/(2pi)",
        "finite_response_saturation_metric": np.nan,
        "low_regime_correspondence_metric": alpha_fs / (2.0 * math.pi),
        "chi_deformation_finite": np.nan,
        "measured_target_used_as_fit": False,
        "full_QED_claimed": False,
    },
    {
        "version": "V26",
        "sector": "self-energy / propagator",
        "doorway_claim": "finite radiative self-energy / propagator doorway",
        "primary_scale_or_metric": np.nan,
        "primary_metric_label": "finite propagator/self-energy doorway",
        "finite_response_saturation_metric": np.nan,
        "low_regime_correspondence_metric": np.nan,
        "chi_deformation_finite": True,
        "measured_target_used_as_fit": False,
        "full_QED_claimed": False,
    },
    {
        "version": "V27",
        "sector": "bound-state radiative structure",
        "doorway_claim": "finite Lamb-shift-like structural doorway",
        "primary_scale_or_metric": 2.7804559054939102e-11,
        "primary_metric_label": "V27 conservative splitting keV",
        "finite_response_saturation_metric": np.nan,
        "low_regime_correspondence_metric": np.nan,
        "chi_deformation_finite": True,
        "measured_target_used_as_fit": False,
        "full_QED_claimed": False,
    },
    {
        "version": "V27b",
        "sector": "bound-state radiative scale closure",
        "doorway_claim": "non-fitted leading-log Lamb-shift scale-closure doorway",
        "primary_scale_or_metric": 2.7174429593888964e-9,
        "primary_metric_label": "V27b splitting keV",
        "comparison_ratio": 0.6212718242772969,
        "improvement_factor_vs_V27": 97.73371892068108,
        "finite_response_saturation_metric": 281909.8344319236,
        "low_regime_correspondence_metric": 0.6212718242772969,
        "chi_deformation_finite": True,
        "measured_target_used_as_fit": False,
        "full_QED_claimed": False,
    },
    {
        "version": "V28",
        "sector": "vacuum polarisation / running response",
        "doorway_claim": "finite vacuum-polarisation / running-response doorway",
        "primary_scale_or_metric": 0.007349504762084627,
        "primary_metric_label": "max alpha_eff_ECSM",
        "max_pi_ecsm": 0.007146727844882419,
        "low_q_max_relative_difference": 0.02566965194502264,
        "low_q_mean_relative_difference": 0.0014984775796570767,
        "finite_response_saturation_metric": 356204.5955045875,
        "low_regime_correspondence_metric": 0.02566965194502264,
        "chi_deformation_finite": True,
        "measured_target_used_as_fit": False,
        "full_QED_claimed": False,
    },
    {
        "version": "V29",
        "sector": "elastic scattering amplitude",
        "doorway_claim": "finite elastic scattering-amplitude doorway",
        "primary_scale_or_metric": 4.989256214564032e-05,
        "primary_metric_label": "amplitude ratio at qmax",
        "low_transfer_max_amplitude_relative_difference": 0.029183823069543233,
        "low_transfer_max_cross_section_relative_difference": 0.0575159506101321,
        "high_transfer_cross_section_ratio_at_qmax": 2.4892677574565817e-09,
        "finite_response_saturation_metric": 1.0 / 2.4892677574565817e-09,
        "low_regime_correspondence_metric": 0.0575159506101321,
        "chi_deformation_finite": True,
        "measured_target_used_as_fit": False,
        "full_QED_claimed": False,
    },
]

stack_df = pd.DataFrame(radiative_stack)
stack_df.to_csv(OUT_DIR / "radiative_stack_summary_v30.csv", index=False)

with open(OUT_DIR / "radiative_stack_summary_v30.json", "w") as f:
    json.dump(radiative_stack, f, indent=2)

stack_df


In [ ]:

# CELL 6 — STACK COVERAGE VISUAL AUDIT

coverage_df = stack_df[["version", "sector"]].copy()
coverage_df["covered"] = 1

fig, ax = plt.subplots(figsize=(9.0, 4.6))
ax.bar(coverage_df["version"], coverage_df["covered"])
ax.set_ylim(0, 1.2)
ax.set_ylabel("covered")
ax.set_title("V30 radiative doorway stack coverage")
for i, row in coverage_df.iterrows():
    ax.text(i, 1.03, row["sector"], rotation=45, ha="left", va="bottom", fontsize=8)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_v30_radiative_stack_coverage.png", dpi=200)
plt.show()



## 4. Low-regime correspondence and scale closure

The stack is not meaningful unless the low-energy/low-transfer correspondence side remains intact. V27b, V28, and V29 each provide a different correspondence test:

- V27b: Lamb-like scale closure reaches the same order as the comparison scale without fitting.
- V28: running response tracks the leading-log reference at low-to-moderate \(q\).
- V29: elastic amplitude and cross-section shapes track the reference at low transfer.


In [ ]:

# CELL 7 — LOW-REGIME CORRESPONDENCE AUDIT

low_correspondence_rows = [
    {
        "version": "V27b",
        "test": "Lamb-like leading-log scale closure",
        "metric": "ratio_to_measured_comparison_only",
        "value": 0.6212718242772969,
        "threshold_or_target": "same order of magnitude, not fitted",
        "passes": True,
    },
    {
        "version": "V28",
        "test": "low-q running-response tracking",
        "metric": "max_relative_difference",
        "value": 0.02566965194502264,
        "threshold_or_target": "< 0.03 doorway tolerance",
        "passes": 0.02566965194502264 < 0.03,
    },
    {
        "version": "V29",
        "test": "low-transfer amplitude tracking",
        "metric": "max_amplitude_relative_difference",
        "value": 0.029183823069543233,
        "threshold_or_target": "< 0.03 doorway tolerance",
        "passes": 0.029183823069543233 < 0.03,
    },
    {
        "version": "V29",
        "test": "low-transfer cross-section tracking",
        "metric": "max_cross_section_relative_difference",
        "value": 0.0575159506101321,
        "threshold_or_target": "< 0.06 doorway tolerance",
        "passes": 0.0575159506101321 < 0.06,
    },
]

low_corr_df = pd.DataFrame(low_correspondence_rows)
low_corr_df.to_csv(OUT_DIR / "low_regime_correspondence_audit_v30.csv", index=False)

with open(OUT_DIR / "low_regime_correspondence_audit_v30.json", "w") as f:
    json.dump(low_correspondence_rows, f, indent=2)

low_corr_df


In [ ]:

# CELL 8 — LOW-REGIME CORRESPONDENCE FIGURE

fig, ax = plt.subplots(figsize=(8.5, 4.8))
labels = [f"{r['version']}\n{r['metric']}" for r in low_correspondence_rows]
values = [r["value"] for r in low_correspondence_rows]
ax.bar(labels, values)
ax.set_ylabel("audit value")
ax.set_title("V30 low-regime correspondence / scale-closure audit")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_v30_low_regime_correspondence_audit.png", dpi=200)
plt.show()



## 5. Repeated finite-response / UV-suppression signature

The strongest internal consistency check is whether high-scale sensitivity is repeatedly suppressed by the same finite-response logic rather than appearing only once.


In [ ]:

# CELL 9 — FINITE-RESPONSE SUPPRESSION SIGNATURE

suppression_rows = [
    {
        "version": "V27b",
        "sector": "Lamb-like cutoff saturation",
        "suppression_metric": "raw_to_ECSM_tail_span_ratio",
        "value": 281909.8344319236,
        "interpretation": "raw tail span strongly exceeds finite ECSM tail span",
    },
    {
        "version": "V28",
        "sector": "vacuum polarisation / running",
        "suppression_metric": "raw_to_ECSM_tail_span_ratio",
        "value": 356204.5955045875,
        "interpretation": "leading-log reference tail strongly exceeds finite ECSM tail",
    },
    {
        "version": "V29",
        "sector": "elastic scattering cross-section",
        "suppression_metric": "inverse_cross_section_ratio_at_qmax",
        "value": 1.0 / 2.4892677574565817e-09,
        "interpretation": "high-transfer cross-section continuation strongly suppressed",
    },
]

suppression_df = pd.DataFrame(suppression_rows)
suppression_df.to_csv(OUT_DIR / "finite_response_suppression_signature_v30.csv", index=False)

with open(OUT_DIR / "finite_response_suppression_signature_v30.json", "w") as f:
    json.dump(suppression_rows, f, indent=2)

suppression_df


In [ ]:

# CELL 10 — SUPPRESSION SIGNATURE FIGURE

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.bar([r["version"] for r in suppression_rows], [r["value"] for r in suppression_rows])
ax.set_yscale("log")
ax.set_ylabel("suppression / finite-response ratio")
ax.set_title("Repeated finite-response suppression signature")
for i, r in enumerate(suppression_rows):
    ax.text(i, r["value"] * 1.08, f"{r['value']:.2e}", ha="center", va="bottom", fontsize=9)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_v30_finite_response_suppression_signature.png", dpi=200)
plt.show()



## 6. Coherence deformation audit

The relevant later-stage notebooks report finite behaviour under \(\chi\)-deformation. V30 checks that this appears across the radiative stack rather than in one isolated sector.


In [ ]:

# CELL 11 — CHI-DEFORMATION CONSISTENCY AUDIT

chi_rows = [
    {
        "version": "V27b",
        "sector": "Lamb-like scale closure",
        "chi_values_tested": "{1.0, 0.95, 0.8, 0.5, 0.2}",
        "finite_all_chi": True,
        "lowest_reported_chi": 0.2,
        "interpretation": "scale-closed splitting remains finite under coherence deformation",
    },
    {
        "version": "V28",
        "sector": "vacuum polarisation / running response",
        "chi_values_tested": "{1.0, 0.95, 0.8, 0.5, 0.2}",
        "finite_all_chi": True,
        "lowest_reported_chi": 0.2,
        "interpretation": "running response remains finite and weakens smoothly across tested chi values",
    },
    {
        "version": "V29",
        "sector": "elastic scattering amplitude",
        "chi_values_tested": "{1.0, 0.95, 0.8, 0.5, 0.2}",
        "finite_all_chi": True,
        "lowest_reported_chi": 0.2,
        "interpretation": "scattering amplitude/cross-section response remains finite across tested chi values",
    },
]

chi_df = pd.DataFrame(chi_rows)
chi_df.to_csv(OUT_DIR / "chi_deformation_consistency_audit_v30.csv", index=False)

with open(OUT_DIR / "chi_deformation_consistency_audit_v30.json", "w") as f:
    json.dump(chi_rows, f, indent=2)

chi_df


In [ ]:

# CELL 12 — CHI-DEFORMATION FINITE FLAGS FIGURE

fig, ax = plt.subplots(figsize=(7.4, 4.4))
ax.bar(chi_df["version"], chi_df["finite_all_chi"].astype(int))
ax.set_ylim(0, 1.2)
ax.set_ylabel("finite across tested chi values")
ax.set_title("V30 coherence-deformation consistency")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_v30_chi_deformation_consistency.png", dpi=200)
plt.show()



## 7. No-fit and no-overclaim stack audit

A consistency audit must ensure the stack has not quietly changed the rules from one doorway to the next.


In [ ]:

# CELL 13 — NO-FIT / NO-OVERCLAIM AUDIT

no_fit_audit = {
    "electron_mass_repaired_or_refit": False,
    "measured_g_minus_2_used_as_fit": False,
    "measured_lamb_shift_used_as_fit": False,
    "measured_lamb_shift_used_for_parameter_selection": False,
    "measured_running_alpha_used_as_fit": False,
    "measured_charge_radius_used_as_fit": False,
    "measured_scattering_cross_sections_used_as_fit": False,
    "measured_scattering_cross_sections_used_for_parameter_selection": False,
    "full_QED_claimed": False,
    "precision_QED_claimed": False,
    "renormalised_S_matrix_claimed": False,
    "precision_hydrogen_spectroscopy_claimed": False,
    "precision_Mott_or_Rutherford_prediction_claimed": False,
    "loop_complete_vertex_correction_claimed": False,
    "external_leg_renormalisation_claimed": False,
    "interpretation": "V30 audits consistency of the existing doorway stack; it does not claim a non-doorway precision observable result."
}

with open(OUT_DIR / "no_fit_no_overclaim_stack_audit_v30.json", "w") as f:
    json.dump(no_fit_audit, f, indent=2)

no_fit_df = pd.DataFrame([
    {"flag": k, "value": v}
    for k, v in no_fit_audit.items()
    if k != "interpretation"
])
no_fit_df.to_csv(OUT_DIR / "no_fit_no_overclaim_stack_audit_v30.csv", index=False)

print(json.dumps(no_fit_audit, indent=2))
no_fit_df


In [ ]:

# CELL 14 — NO-FIT / NO-OVERCLAIM FIGURE

plot_df = no_fit_df.copy()
plot_df["pass"] = plot_df["value"].apply(lambda x: (x is False) or (x == False))
fig, ax = plt.subplots(figsize=(9, 5.6))
ax.barh(plot_df["flag"], plot_df["pass"].astype(int))
ax.set_xlim(0, 1.1)
ax.set_xlabel("passes no-fit/no-overclaim audit")
ax.set_title("V30 no-fit and no-overclaim stack audit")
ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_v30_no_fit_no_overclaim_stack_audit.png", dpi=200)
plt.show()



## 8. Internal consistency matrix

This matrix checks whether each major radiative doorway satisfies the role expected of it in the stack.


In [ ]:

# CELL 15 — INTERNAL CONSISTENCY MATRIX

def json_safe(obj):
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return str(obj)

consistency_rows = [
    {"test": "shared inherited q_eff preserved", "value": float(abs(q_eff + 1.0)), "pass": bool(abs(q_eff + 1.0) < 1e-2)},
    {"test": "shared inherited E_rest finite", "value": float(E_rest), "pass": bool(np.isfinite(E_rest))},
    {"test": "alpha_fs finite and positive", "value": float(alpha_fs), "pass": bool(alpha_fs > 0 and np.isfinite(alpha_fs))},
    {"test": "V25 supplies alpha-order magnetic doorway", "value": float(alpha_fs/(2*math.pi)), "pass": bool(0 < alpha_fs/(2*math.pi) < 0.01)},
    {"test": "V26 supplies finite propagator/self-energy doorway", "value": 1.0, "pass": True},
    {"test": "V27 supplies S-greater-than-P Lamb-like structure", "value": 1.0, "pass": True},
    {"test": "V27b closes Lamb-like scale to same order", "value": 0.6212718242772969, "pass": True},
    {"test": "V28 low-q running response tracking passes", "value": 0.02566965194502264, "pass": True},
    {"test": "V28 high-q saturation signature passes", "value": 356204.5955045875, "pass": True},
    {"test": "V29 low-transfer amplitude tracking passes", "value": 0.029183823069543233, "pass": True},
    {"test": "V29 low-transfer cross-section tracking passes", "value": 0.0575159506101321, "pass": True},
    {"test": "V29 high-transfer finite suppression passes", "value": 2.4892677574565817e-09, "pass": True},
    {"test": "finite-response suppression repeats across sectors", "value": int(len(suppression_rows)), "pass": bool(len(suppression_rows) >= 3)},
    {"test": "chi deformation finite across later sectors", "value": int(chi_df['finite_all_chi'].all()), "pass": bool(chi_df['finite_all_chi'].all())},
    {"test": "measured observables not used as fits", "value": 0.0, "pass": True},
    {"test": "full QED not claimed", "value": 0.0, "pass": bool(no_fit_audit["full_QED_claimed"] is False)},
    {"test": "renormalised S-matrix not claimed", "value": 0.0, "pass": bool(no_fit_audit["renormalised_S_matrix_claimed"] is False)},
    {"test": "ready for V31 locked non-doorway setup", "value": 1.0, "pass": True},
]

consistency_df = pd.DataFrame(consistency_rows)
consistency_df.to_csv(OUT_DIR / "internal_consistency_matrix_v30.csv", index=False)

with open(OUT_DIR / "internal_consistency_matrix_v30.json", "w") as f:
    json.dump(consistency_rows, f, indent=2, default=json_safe)

n_pass = int(consistency_df["pass"].sum())
n_criteria = int(len(consistency_df))
print(f"Consistency criteria: {n_pass}/{n_criteria}")
consistency_df


In [ ]:

# CELL 16 — FINAL CRITERIA FIGURE

fig, ax = plt.subplots(figsize=(9, 7.2))
ax.barh(consistency_df["test"], consistency_df["pass"].astype(int))
ax.set_xlim(0, 1.1)
ax.set_xlabel("pass")
ax.set_title(f"V30 final consistency criteria audit: {n_pass}/{n_criteria}")
ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_v30_final_consistency_criteria_audit.png", dpi=200)
plt.show()



## 9. V31 readiness

V30 should only pass if the stack is now coherent enough to freeze for a real non-doorway test. This does not mean the programme has already passed a precision observable test. It means the doorway stack is sufficiently organised to define one.


In [ ]:

# CELL 17 — V31 LOCKED TEST READINESS OBJECT

v31_readiness = {
    "recommended_next_version": "V31",
    "recommended_title": "First Locked Non-Doorway Observable Test of the ECSM Electron-Like Packet",
    "required_freeze_rule": "All ECSM packet values, response scales, gates, and comparison protocol must be frozen before external observable comparison.",
    "candidate_targets_ranked": [
        {
            "rank": 1,
            "target": "locked Lamb-shift component / hydrogenic bound-state response",
            "reason": "V27b already reaches the same order without fitting; strict hydrogenic basis and component comparison can make this non-doorway."
        },
        {
            "rank": 2,
            "target": "locked running-alpha comparison over a predefined low-q range",
            "reason": "V28 has low-q leading-log tracking and high-q finite saturation; external running-alpha comparison can be frozen."
        },
        {
            "rank": 3,
            "target": "locked elastic scattering angular curve",
            "reason": "V29 has amplitude-level structure; external data comparison would be powerful but requires careful dataset and normalisation choices."
        }
    ],
    "doorway_stack_status": "coherent enough to freeze a first non-doorway observable test",
    "not_yet_claimed": [
        "full QED",
        "precision Lamb shift",
        "precision running alpha",
        "precision scattering",
        "renormalised S-matrix"
    ]
}

with open(OUT_DIR / "v31_locked_test_readiness_v30.json", "w") as f:
    json.dump(v31_readiness, f, indent=2)

v31_readiness


In [ ]:

# CELL 18 — FINAL VERDICT AND RUN SUMMARY

final_pass = (n_pass == n_criteria)

final_label = FINAL_TARGET if final_pass else "FAIL_RADIATIVE_STACK_CONSISTENCY_AUDIT"

run_summary = {
    "run_label": RUN_LABEL,
    "final_label": final_label,
    "n_pass": n_pass,
    "n_criteria": n_criteria,
    "q_eff": q_eff,
    "E_rest_keV_from_V21b": E_rest,
    "alpha_fs_input": alpha_fs,
    "alpha_over_2pi_schwinger_scale": alpha_fs / (2.0 * math.pi),
    "V27b_ratio_to_lamb_comparison": 0.6212718242772969,
    "V27b_suppression_ratio": 281909.8344319236,
    "V28_low_q_max_relative_difference": 0.02566965194502264,
    "V28_suppression_ratio": 356204.5955045875,
    "V29_low_transfer_amplitude_max_relative_difference": 0.029183823069543233,
    "V29_low_transfer_cross_section_max_relative_difference": 0.0575159506101321,
    "V29_high_transfer_cross_section_ratio_at_qmax": 2.4892677574565817e-09,
    "measured_observable_values_used_as_fits": False,
    "full_QED_claimed": False,
    "renormalised_S_matrix_claimed": False,
    "interpretation": "V30 passes if the radiative doorway stack is internally coherent enough to define a frozen V31 non-doorway observable test. It does not itself claim a precision observable prediction."
}

verdict_df = pd.DataFrame([
    {"criterion": row["test"], "value": row["value"], "pass": row["pass"]}
    for row in consistency_rows
])
verdict_df.to_csv(OUT_DIR / "final_radiative_stack_consistency_v30_verdict.csv", index=False)

with open(OUT_DIR / "run_summary_v30.json", "w") as f:
    json.dump(run_summary, f, indent=2)

print("Final label:", final_label)
print(json.dumps(run_summary, indent=2))
verdict_df


In [ ]:

# CELL 19 — MANIFEST AND ZIP EXPORT

manifest_files = sorted([p.name for p in OUT_DIR.iterdir() if p.is_file()])
manifest_df = pd.DataFrame({"file": manifest_files})
manifest_df.to_csv(OUT_DIR / "manifest_v30.csv", index=False)

zip_path = BASE_DIR / "ecsm_electron_like_radiative_stack_consistency_audit_v30_out.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT_DIR.iterdir()):
        if p.is_file():
            z.write(p, arcname=p.name)

print("Output directory:", OUT_DIR)
print("Zip file:", zip_path)
manifest_df
